# 03 MongoDB Atlas NoSQL Design, CRUD, Aggregation and Optimisation
This notebook builds a MongoDB Atlas document model for NorthStar. It creates customer case documents with nested delivery, complaint and incident records, then demonstrates CRUD, aggregation, indexing and explain plans.

In [ ]:
!pip -q install pymongo[srv] dnspython pandas python-dotenv

In [ ]:
import os
from pathlib import Path
import pandas as pd
from pymongo import MongoClient, ASCENDING, DESCENDING

# In Colab, store MONGODB_URI in Colab Secrets, or paste it temporarily here.
# Never commit a real MongoDB URI to GitHub.
MONGODB_URI = os.getenv("MONGODB_URI") or "PASTE_YOUR_MONGODB_ATLAS_URI_HERE"
DB_NAME = "northstar_analytics"
DATA_DIR = Path("/content/northstar_dataset")

if MONGODB_URI == "PASTE_YOUR_MONGODB_ATLAS_URI_HERE":
    raise ValueError("Paste your MongoDB Atlas URI or set it in Colab Secrets as MONGODB_URI.")
client = MongoClient(MONGODB_URI)
db = client[DB_NAME]
print(client.admin.command("ping"))

In [ ]:
orders = pd.read_csv(DATA_DIR / "orders.csv")
deliveries = pd.read_csv(DATA_DIR / "deliveries.csv")
complaints = pd.read_csv(DATA_DIR / "complaints.csv")
incidents = pd.read_csv(DATA_DIR / "incidents.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")
app_events = pd.read_csv(DATA_DIR / "app_events.csv")
print(orders.shape, deliveries.shape, complaints.shape, incidents.shape, customers.shape, app_events.shape)

In [ ]:
for name in ["customer_cases", "customers", "app_events"]:
    db[name].drop()

db.customers.insert_many(customers.to_dict("records"))
db.app_events.insert_many(app_events.to_dict("records"))

case_docs = []
for _, order in orders.iterrows():
    related_delivery = deliveries[deliveries["order_id"] == order["order_id"]]
    delivery_doc = related_delivery.iloc[0].to_dict() if len(related_delivery) else None
    delivery_id = delivery_doc.get("delivery_id") if delivery_doc else None
    case_docs.append({
        "order_id": order["order_id"],
        "customer_id": order["customer_id"],
        "service_type": order["service_type"],
        "priority_level": order["priority_level"],
        "booking_channel": order["booking_channel"],
        "pickup_zone": str(order["pickup_zone"]).title(),
        "dropoff_zone": str(order["dropoff_zone"]).title(),
        "order_value": float(order["order_value"]),
        "promised_window_hours": int(order["promised_window_hours"]),
        "special_handling_flag": int(order["special_handling_flag"]),
        "delivery": delivery_doc,
        "complaints": complaints[complaints["order_id"] == order["order_id"]].to_dict("records"),
        "incidents": incidents[incidents["delivery_id"] == delivery_id].to_dict("records") if delivery_id else []
    })

db.customer_cases.insert_many(case_docs)
print("customer_cases:", db.customer_cases.count_documents({}))
print("customers:", db.customers.count_documents({}))
print("app_events:", db.app_events.count_documents({}))

In [ ]:
# CREATE example
new_doc = {
    "order_id": "TEST_ORDER_001",
    "customer_id": "TEST_CUSTOMER",
    "service_type": "Parcel",
    "priority_level": "High",
    "booking_channel": "App",
    "pickup_zone": "Central",
    "dropoff_zone": "North",
    "order_value": 99.99,
    "promised_window_hours": 4,
    "special_handling_flag": 1,
    "delivery": {"delivery_status": "Delayed", "hub_id": "H05"},
    "complaints": [],
    "incidents": []
}
db.customer_cases.insert_one(new_doc)
print(db.customer_cases.find_one({"order_id": "TEST_ORDER_001"}, {"_id": 0}))

In [ ]:
# READ example
failed_cases = list(db.customer_cases.find({"delivery.delivery_status": "Failed"}, {"_id": 0, "order_id": 1, "customer_id": 1, "service_type": 1, "delivery.delivery_status": 1}).limit(5))
failed_cases

In [ ]:
# UPDATE example
db.customer_cases.update_one({"order_id": "TEST_ORDER_001"}, {"$set": {"review_status": "checked_for_coursework_demo"}})
print(db.customer_cases.find_one({"order_id": "TEST_ORDER_001"}, {"_id": 0, "order_id": 1, "review_status": 1}))

In [ ]:
# DELETE example
delete_result = db.customer_cases.delete_one({"order_id": "TEST_ORDER_001"})
print("Deleted:", delete_result.deleted_count)

In [ ]:
# Aggregation: delayed/failed rates by hub
pipeline = [
    {"$match": {"delivery": {"$ne": None}}},
    {"$group": {
        "_id": "$delivery.hub_id",
        "total": {"$sum": 1},
        "problem_deliveries": {"$sum": {"$cond": [{"$in": ["$delivery.delivery_status", ["Delayed", "Failed"]]}, 1, 0]}},
        "avg_rating": {"$avg": "$delivery.customer_rating_post_delivery"},
        "avg_cost": {"$avg": "$delivery.fuel_or_charge_cost"}
    }},
    {"$addFields": {"problem_rate": {"$divide": ["$problem_deliveries", "$total"]}}},
    {"$sort": {"problem_rate": -1}}
]
list(db.customer_cases.aggregate(pipeline))

In [ ]:
# Explain before indexes or after dropping optional indexes
query = {"delivery.delivery_status": "Failed"}
print(db.command("explain", {"find": "customer_cases", "filter": query}, verbosity="executionStats"))

In [ ]:
# Create indexes for common operational queries
db.customer_cases.create_index([("customer_id", ASCENDING)])
db.customer_cases.create_index([("service_type", ASCENDING), ("pickup_zone", ASCENDING)])
db.customer_cases.create_index([("delivery.delivery_status", ASCENDING)])
db.customer_cases.create_index([("delivery.hub_id", ASCENDING), ("delivery.delivery_status", ASCENDING)])
db.customer_cases.create_index([("complaints.complaint_type", ASCENDING)])
db.app_events.create_index([("customer_id", ASCENDING), ("event_timestamp", DESCENDING)])
db.app_events.create_index([("event_type", ASCENDING), ("success_flag", ASCENDING)])
print(list(db.customer_cases.list_indexes()))

In [ ]:
# Explain after indexing. Look for IXSCAN in the winningPlan and lower totalDocsExamined.
print(db.command("explain", {"find": "customer_cases", "filter": {"delivery.delivery_status": "Failed"}}, verbosity="executionStats"))